# Object Detection API Demo

<table align="left"><td>
  <a target="_blank"  href="https://colab.sandbox.google.com/github/tensorflow/models/blob/master/research/object_detection/colab_tutorials/object_detection_tutorial.ipynb">
    <img src="https://www.tensorflow.org/images/colab_logo_32px.png" />Run in Google Colab
  </a>
</td><td>
  <a target="_blank"  href="https://github.com/tensorflow/models/blob/master/research/object_detection/colab_tutorials/object_detection_tutorial.ipynb">
    <img width=32px src="https://www.tensorflow.org/images/GitHub-Mark-32px.png" />View source on GitHub</a>
</td></table>

Welcome to the [Object Detection API](https://github.com/tensorflow/models/tree/master/research/object_detection). This notebook will walk you step by step through the process of using a pre-trained model to detect objects in an image.

> **Important**: This tutorial is to help you through the first step towards using [Object Detection API](https://github.com/tensorflow/models/tree/master/research/object_detection) to build models. If you just just need an off the shelf model that does the job, see the [TFHub object detection example](https://colab.sandbox.google.com/github/tensorflow/docs/blob/master/site/en/hub/tutorials/object_detection.ipynb).

# Setup

Important: If you're running on a local machine, be sure to follow the [installation instructions](https://github.com/tensorflow/models/blob/master/research/object_detection/g3doc/tf2.md). This notebook includes only what's necessary to run in Colab.

### Install

In [3]:
# !pip install -U --pre tensorflow=="2.*"
# !pip install tf_slim

Make sure you have `pycocotools` installed

In [4]:
# !pip install pycocotools

Get `tensorflow/models` or `cd` to parent directory of the repository.

In [5]:
# import os
# import pathlib


# if "models" in pathlib.Path.cwd().parts:
#   while "models" in pathlib.Path.cwd().parts:
#     os.chdir('..')
# elif not pathlib.Path('models').exists():
#   !git clone --depth 1 https://github.com/tensorflow/models

Compile protobufs and install the object_detection package

In [6]:
# %%bash
# cd models/research/
# protoc object_detection/protos/*.proto --python_out=.

In [7]:
# %%bash 
# cd models/research
# pip install .

### Imports

In [8]:
import numpy as np
import os
import six.moves.urllib as urllib
import sys
import tarfile
import tensorflow as tf
import zipfile
import pathlib

from collections import defaultdict
from io import StringIO
from matplotlib import pyplot as plt
from PIL import Image
from IPython.display import display

e:\conda_envs\tensorflow_obj\lib\site-packages\google\api_core\_python_version_support.py:246: FutureWarning: You are using a non-supported Python version (3.9.25). Google will not post any further updates to google.api_core supporting this Python version. Please upgrade to the latest Python version, or at least Python 3.10, and then update google.api_core.
  warnings.warn(message, FutureWarning)
e:\conda_envs\tensorflow_obj\lib\site-packages\google\auth\__init__.py:54: FutureWarning: You are using a Python version 3.9 past its end of life. Google will update google-auth with critical bug fixes on a best-effort basis, but not with any other fixes or features. Please upgrade your Python version, and then update google-auth.
  warnings.warn(eol_message.format("3.9"), FutureWarning)
e:\conda_envs\tensorflow_obj\lib\site-packages\google\oauth2\__init__.py:40: FutureWarning: You are using a Python version 3.9 past its end of life. Google will update google-auth with critical bug fixes on a 

In [9]:
import cv2

Import the object detection module.

In [10]:
from object_detection.utils import ops as utils_ops
from object_detection.utils import label_map_util
from object_detection.utils import visualization_utils as vis_util

Patches:

In [11]:
# patch tf1 into `utils.ops`
utils_ops.tf = tf.compat.v1

# Patch the location of gfile
tf.gfile = tf.io.gfile

# Model preparation 

## Variables

Any model exported using the `export_inference_graph.py` tool can be loaded here simply by changing the path.

By default we use an "SSD with Mobilenet" model here. See the [detection model zoo](https://github.com/tensorflow/models/blob/master/research/object_detection/g3doc/f1_detection_zoo.md) for a list of other models that can be run out-of-the-box with varying speeds and accuracies.

## Loader

In [12]:
def load_model(model_name):
  base_url = 'http://download.tensorflow.org/models/object_detection/'
  model_file = model_name + '.tar.gz'
  model_dir = tf.keras.utils.get_file(
    fname=model_name, 
    origin=base_url + model_file,
    untar=True)

  model_dir = pathlib.Path(model_dir)/"saved_model"

  model = tf.saved_model.load(str(model_dir))

  return model

## Loading label map
Label maps map indices to category names, so that when our convolution network predicts `5`, we know that this corresponds to `airplane`.  Here we use internal utility functions, but anything that returns a dictionary mapping integers to appropriate string labels would be fine

In [13]:
# List of the strings that is used to add correct label for each box.
PATH_TO_LABELS = os.path.join(os.path.abspath('.'), 'Detection项目\Detection项目\models\models\\research\object_detection\data\mscoco_label_map.pbtxt')
print(PATH_TO_LABELS)
category_index = label_map_util.create_category_index_from_labelmap(PATH_TO_LABELS, use_display_name=True)

e:\a学习\交通工程综合设计\Detection项目\Detection项目\models\models\research\object_detection\data\mscoco_label_map.pbtxt


For the sake of simplicity we will test on 3 images:

In [14]:
# If you want to test the code with your images, just add path to the images to the TEST_IMAGE_PATHS.
PATH_TO_TEST_IMAGES_DIR = pathlib.Path(os.path.join(os.path.abspath('.'), 'Detection项目\Detection项目\models\models\\research\object_detection\\test_images'))
print(PATH_TO_TEST_IMAGES_DIR)
TEST_IMAGE_PATHS = sorted(list(PATH_TO_TEST_IMAGES_DIR.glob("*.jpg")))
TEST_IMAGE_PATHS

e:\a学习\交通工程综合设计\Detection项目\Detection项目\models\models\research\object_detection\test_images


[WindowsPath('e:/a学习/交通工程综合设计/Detection项目/Detection项目/models/models/research/object_detection/test_images/image1.jpg'),
 WindowsPath('e:/a学习/交通工程综合设计/Detection项目/Detection项目/models/models/research/object_detection/test_images/image2.jpg'),
 WindowsPath('e:/a学习/交通工程综合设计/Detection项目/Detection项目/models/models/research/object_detection/test_images/image3.jpg')]

# Detection

Load an object detection model:

In [15]:
model_name = 'ssd_mobilenet_v1_coco_2017_11_17'
detection_model = load_model(model_name)

INFO:tensorflow:Saver not created because there are no variables in the graph to restore


Check the model's input signature, it expects a batch of 3-color images of type uint8:

In [16]:
print(detection_model.signatures['serving_default'].inputs)

[<tf.Tensor 'image_tensor:0' shape=(None, None, None, 3) dtype=uint8>]


And returns several outputs:

In [17]:
detection_model.signatures['serving_default'].output_dtypes

{'num_detections': tf.float32,
 'detection_boxes': tf.float32,
 'detection_scores': tf.float32,
 'detection_classes': tf.float32}

In [18]:
detection_model.signatures['serving_default'].output_shapes

{'num_detections': TensorShape([None]),
 'detection_boxes': TensorShape([None, 100, 4]),
 'detection_scores': TensorShape([None, 100]),
 'detection_classes': TensorShape([None, 100])}

Add a wrapper function to call the model, and cleanup the outputs:

In [19]:
def run_inference_for_single_image(model, image):
  image = np.asarray(image)
  # The input needs to be a tensor, convert it using `tf.convert_to_tensor`.
  input_tensor = tf.convert_to_tensor(image)
  # The model expects a batch of images, so add an axis with `tf.newaxis`.
  input_tensor = input_tensor[tf.newaxis,...]

  # Run inference
  model_fn = model.signatures['serving_default']
  output_dict = model_fn(input_tensor)

  # All outputs are batches tensors.
  # Convert to numpy arrays, and take index [0] to remove the batch dimension.
  # We're only interested in the first num_detections.
  num_detections = int(output_dict.pop('num_detections'))
  output_dict = {key:value[0, :num_detections].numpy() 
                 for key,value in output_dict.items()}
  output_dict['num_detections'] = num_detections

  # detection_classes should be ints.
  output_dict['detection_classes'] = output_dict['detection_classes'].astype(np.int64)
   
  # Handle models with masks:
  if 'detection_masks' in output_dict:
    # Reframe the the bbox mask to the image size.
    detection_masks_reframed = utils_ops.reframe_box_masks_to_image_masks(
              output_dict['detection_masks'], output_dict['detection_boxes'],
               image.shape[0], image.shape[1])      
    detection_masks_reframed = tf.cast(detection_masks_reframed > 0.5,
                                       tf.uint8)
    output_dict['detection_masks_reframed'] = detection_masks_reframed.numpy()
    
  return output_dict

Run it on each test image and show the results:

In [22]:
import cv2
import os
import numpy as np

def process_video(model, input_video_path, output_video_path, score_thresh=0.5):
    cap = cv2.VideoCapture(input_video_path)

    if not cap.isOpened():
        print(f"无法打开视频: {input_video_path}")
        return

    # 读取视频信息
    fps = cap.get(cv2.CAP_PROP_FPS)
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    print(f"视频FPS: {fps}")
    print(f"分辨率: {width} x {height}")
    print(f"总帧数: {total_frames}")

    # 创建输出文件夹
    os.makedirs(os.path.dirname(output_video_path) if os.path.dirname(output_video_path) else ".", exist_ok=True)

    # 定义视频编码器
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out = cv2.VideoWriter(output_video_path, fourcc, fps, (width, height))

    frame_id = 0

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        frame_id += 1
        print(f"正在处理第 {frame_id}/{total_frames} 帧", end='\r')

        # OpenCV读进来是BGR，模型一般按RGB处理
        frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

        # 推理
        output_dict = run_inference_for_single_image(model, frame_rgb)

        # 在RGB图上画框
        vis_util.visualize_boxes_and_labels_on_image_array(
            frame_rgb,
            output_dict['detection_boxes'],
            output_dict['detection_classes'],
            output_dict['detection_scores'],
            category_index,
            instance_masks=output_dict.get('detection_masks_reframed', None),
            use_normalized_coordinates=True,
            min_score_thresh=score_thresh,
            line_thickness=4
        )

        # 再转回BGR写入视频
        result_bgr = cv2.cvtColor(frame_rgb, cv2.COLOR_RGB2BGR)
        out.write(result_bgr)

    cap.release()
    out.release()
    print(f"\n处理完成，结果已保存到: {output_video_path}")

In [25]:
input_video_path = "Detection项目\Detection项目\models\models\\research\object_detection\\test_images\\video1.AVI"
output_video_path = "output_detected1.AVI"

process_video(detection_model, input_video_path, output_video_path)

视频FPS: 30.00030000300003
分辨率: 320 x 240
总帧数: 1828
正在处理第 1823/1828 帧
处理完成，结果已保存到: output_detected1.AVI
